# 5.1 端侧NPU适配

端侧NPU（Neural Processing Unit）是专为神经网络推理设计的加速器，具有高算力、低功耗的特点。不同厂商的NPU架构和SDK不同，需要针对性适配。

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### NPU算子兼容性分析

不同NPU支持的算子集合不同，需要检查模型中的算子是否被目标NPU支持，不支持的算子需要回退到CPU执行。

In [ ]:
class NPUCompatibilityChecker:
    """NPU算子兼容性检查器
    注意：本实现为教学演示，使用硬编码的算子支持列表。
    实际NPU适配需使用各厂商SDK（如QNN、Core ML Tools）进行真实的算子验证和模型转换。"""
    def __init__(self):
        self.npu_support = {
            'qualcomm_hexagon': {
                'supported': ['conv2d', 'linear', 'relu', 'gelu', 'layer_norm',
                              'batch_norm', 'softmax', 'add', 'mul', 'matmul',
                              'reshape', 'transpose', 'concat', 'split',
                              'quantize', 'dequantize'],
                'unsupported': ['attention', 'rope', 'topk', 'multinomial'],
                'precision': ['int8', 'int4', 'fp16'],
                'tops': 45,
            },
            'apple_ane': {
                'supported': ['conv2d', 'linear', 'relu', 'gelu', 'layer_norm',
                              'batch_norm', 'softmax', 'add', 'mul', 'matmul'],
                'unsupported': ['attention', 'rope', 'topk', 'quantize_int4'],
                'precision': ['fp16', 'int8'],
                'tops': 38,
            },
            'mediatek_apu': {
                'supported': ['conv2d', 'linear', 'relu', 'gelu', 'layer_norm',
                              'softmax', 'add', 'mul', 'matmul'],
                'unsupported': ['attention', 'rope', 'topk', 'multinomial', 'int4'],
                'precision': ['int8', 'int16', 'fp16'],
                'tops': 30,
            },
        }

    def check_model(self, model_ops, target_npu):
        """检查模型算子对目标NPU的兼容性"""
        npu_info = self.npu_support[target_npu]
        results = {'supported': [], 'unsupported': [], 'fallback': []}
        for op in model_ops:
            if op in npu_info['supported']:
                results['supported'].append(op)
            elif op in npu_info['unsupported']:
                results['unsupported'].append(op)
                results['fallback'].append((op, 'CPU'))
            else:
                results['unsupported'].append(op)
                results['fallback'].append((op, 'Unknown'))
        return results

checker = NPUCompatibilityChecker()
llm_ops = ['linear', 'layer_norm', 'gelu', 'attention', 'rope',
           'softmax', 'add', 'mul', 'reshape', 'transpose', 'topk']

print(f"=== LLM算子NPU兼容性分析 ===")
for npu_name in ['qualcomm_hexagon', 'apple_ane', 'mediatek_apu']:
    info = checker.npu_support[npu_name]
    result = checker.check_model(llm_ops, npu_name)
    print(f"\n--- {npu_name} ({info['tops']} TOPS) ---")
    print(f"  支持精度: {info['precision']}")
    print(f"  支持算子: {len(result['supported'])}/{len(llm_ops)}")
    print(f"  不支持算子: {result['unsupported']}")
    if result['fallback']:
        print(f"  需回退CPU: {[(op, dev) for op, dev in result['fallback']]}")

print(f"\n=== NPU适配关键挑战 ===")
print(f"1. Attention算子: 大多数NPU不支持，需拆分为QKV投影+Softmax+Matmul")
print(f"2. RoPE: 需自定义实现或回退CPU")
print(f"3. 动态形状: NPU通常只支持静态形状，需padding")
print(f"4. INT4支持: 仅高通Hexagon支持，其他NPU需INT8")

### NPU量化适配

In [ ]:
class NPUQuantizationAdapter:
    """NPU量化适配器: 将模型量化为NPU支持的格式"""
    def __init__(self, target_npu='qualcomm_hexagon'):
        self.target_npu = target_npu
        self.supported_precisions = {
            'qualcomm_hexagon': {'weight': ['int8', 'int4'], 'activation': ['int8', 'fp16']},
            'apple_ane': {'weight': ['int8', 'fp16'], 'activation': ['fp16']},
            'mediatek_apu': {'weight': ['int8', 'int16'], 'activation': ['int8', 'fp16']},
        }

    def get_optimal_config(self, model_size_mb, memory_budget_mb):
        """根据模型大小和内存预算推荐量化配置"""
        precisions = self.supported_precisions[self.target_npu]
        configs = []

        for w_prec in precisions['weight']:
            for a_prec in precisions['activation']:
                if w_prec == 'int4':
                    w_ratio = 0.25
                elif w_prec == 'int8':
                    w_ratio = 0.5
                elif w_prec == 'int16':
                    w_ratio = 1.0
                else:
                    w_ratio = 0.5

                estimated_size = model_size_mb * w_ratio
                fits = estimated_size <= memory_budget_mb
                configs.append({
                    'weight': w_prec, 'activation': a_prec,
                    'estimated_size_mb': estimated_size,
                    'fits_budget': fits
                })

        return sorted(configs, key=lambda x: (-x['fits_budget'], x['estimated_size_mb']))

adapter = NPUQuantizationAdapter('qualcomm_hexagon')
configs = adapter.get_optimal_config(model_size_mb=4000, memory_budget_mb=2000)

print(f"=== 高通Hexagon NPU量化配置推荐 ===")
print(f"模型大小: 4000MB (FP32), 内存预算: 2000MB")
print(f"\n{'权重精度':<12} {'激活精度':<12} {'估计大小(MB)':<15} {'是否适合':<10}")
print("-" * 49)
for cfg in configs:
    print(f"{cfg['weight']:<12} {cfg['activation']:<12} {cfg['estimated_size_mb']:<15.0f} {'✓' if cfg['fits_budget'] else '✗':<10}")

print(f"\n推荐: W4A8 (权重INT4 + 激活INT8) - 最小内存占用 + NPU原生支持")